<img src="image.png" alt="My Image" width="1000">

### ============================================================
### STEP 1: DATA LOADING, MERGING & EDA
### ============================================================

In [46]:
import numpy as np
import pandas as pd

In [47]:
# 1. Load the two separate CSV files
fake_df = pd.read_csv("Fake.csv")
true_df = pd.read_csv("True.csv")

# 2. Add target label column
fake_df["label"] = "fake"
true_df["label"] = "true"

# 3. Combine both datasets into a single DataFrame
df = pd.concat([fake_df, true_df], axis=0).reset_index(drop=True)

# 4. Display Dataset Shape
print("=== DATASET OVERVIEW ===")
print("Total Records & Columns:", df.shape)

=== DATASET OVERVIEW ===
Total Records & Columns: (44898, 5)


In [48]:
# 5. Check for Null / Missing Values
print("\n=== MISSING VALUES ===")
print(df.isna().sum())

# 6. Check for Duplicate Rows
print("\n=== DUPLICATE ROWS ===")
print("Total Duplicates:", df.duplicated().sum())


=== MISSING VALUES ===
title      0
text       0
subject    0
date       0
label      0
dtype: int64

=== DUPLICATE ROWS ===
Total Duplicates: 209


In [49]:
# 7. Check Class Distribution (Target Balance)
print("\n=== CLASS DISTRIBUTION ===")
print(df["label"].value_counts())



=== CLASS DISTRIBUTION ===
label
fake    23481
true    21417
Name: count, dtype: int64


In [50]:
# 8. Preview First 5 Rows
print("\n=== DATASET PREVIEW (HEAD) ===")
print(df.head())



=== DATASET PREVIEW (HEAD) ===
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date label  
0  December 31, 2017  fake  
1  December 31, 2017  fake  
2  December 30, 2017  fake  
3  December 29, 2017  fake  
4  December 25, 2017  fake  


### =========================================================
### STEP 2: FEATURE SELECTION & TARGET LABEL ENCODING
### =========================================================

In [51]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

In [52]:
# 1. Separate Feature (X) and Target (y)
X = df["text"]
y = df["label"]

# 2. Initialize LabelEncoder
encoder = LabelEncoder()
encoder

LabelEncoder()

In [53]:
# 3. Fit and transform target column ('fake' -> 0, 'true' -> 1)
y_encoded = encoder.fit_transform(y)

# 4. Display Label Mapping Details
print("=== TARGET ENCODING SUMMARY ===")
print("Encoded Class Labels:", encoder.classes_)
print("Mapping: 0 ->", encoder.classes_[0], "| 1 ->", encoder.classes_[1])
print("\nFirst 5 Encoded Target Values:")
print(y_encoded[:5])


=== TARGET ENCODING SUMMARY ===
Encoded Class Labels: ['fake' 'true']
Mapping: 0 -> fake | 1 -> true

First 5 Encoded Target Values:
[0 0 0 0 0]


### =========================================================
### STEP 3: STRATIFIED TRAIN-TEST SPLIT (80% Train / 20% Test)
### =========================================================

In [54]:
from sklearn.model_selection import train_test_split

In [55]:
# Split the dataset using stratify to maintain class ratio
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Display Split Summary
print("=== TRAIN-TEST SPLIT SUMMARY ===")
print("Training set samples (X_train):", len(X_train))
print("Testing set samples (X_test):", len(X_test))
print("Total dataset records:  ", len(X))


=== TRAIN-TEST SPLIT SUMMARY ===
Training set samples (X_train): 35918
Testing set samples (X_test): 8980
Total dataset records:   44898


### =========================================================
### STEP 4: REGEX TEXT PREPROCESSING (clean_text)
### =========================================================

In [56]:
import re

def clean_text(text):
    if not isinstance(text, str):
        text = str(text)
    
    # Convert to lowercase
    text = text.lower()
    
    # Remove non-alphabet characters
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Replace multiple spaces with a single space and strip leading/trailing spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply cleaning function to both Training and Testing sets
X_train_clean = X_train.apply(clean_text)
X_test_clean = X_test.apply(clean_text)


In [57]:
# Display a sample before and after cleaning
print("\n=== REGEX PREPROCESSING SAMPLE ===")
print("Original Text (Sample 1):")
print(X_train.iloc[0][:150], "...")
print("\nCleaned Text (Sample 1):")
print(X_train_clean.iloc[0][:150], "...")


=== REGEX PREPROCESSING SAMPLE ===
Original Text (Sample 1):
PARIS (Reuters) - French bank Societe Generale on Wednesday rejected the far-right National Front s accusations that it had acted to  suffocate  the p ...

Cleaned Text (Sample 1):
paris reuters french bank societe generale on wednesday rejected the farright national front s accusations that it had acted to suffocate the party by ...


### =========================================================
### STEP 5: FEATURE EXTRACTION (CountVectorizer / Bag of Words)
### =========================================================

In [58]:
from sklearn.feature_extraction.text import CountVectorizer


# 1. Initialize CountVectorizer
cv = CountVectorizer()
cv

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


In [59]:
# 2. Fit and transform training text data
X_train_cv = cv.fit_transform(X_train_clean)

# 3. Transform testing text data (transform only, do NOT fit)
X_test_cv = cv.transform(X_test_clean)

# Display Vectorization Summary
print("=== FEATURE EXTRACTION (BAG OF WORDS) SUMMARY ===")
print("Training feature matrix shape:", X_train_cv.shape)
print("Testing feature matrix shape: ", X_test_cv.shape)
print("Total unique vocabulary words (features):", len(cv.get_feature_names_out()))


=== FEATURE EXTRACTION (BAG OF WORDS) SUMMARY ===
Training feature matrix shape: (35918, 190178)
Testing feature matrix shape:  (8980, 190178)
Total unique vocabulary words (features): 190178


In [60]:
cv

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"
,analyzer,'word'


### =========================================================
### STEP 6: MODEL TRAINING & COMPARISON
### =========================================================

In [61]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier


In [62]:
# 1. Initialize the three classification models
model_nb = MultinomialNB()
model_knn = KNeighborsClassifier()
model_dt = DecisionTreeClassifier()

In [63]:
# 2. Train models on vectorized training data
print("Training Multinomial Naive Bayes model...")
model_nb.fit(X_train_cv, y_train)
model_nb

Training Multinomial Naive Bayes model...


,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


In [64]:
print("Training K-Nearest Neighbors model...")
model_knn.fit(X_train_cv, y_train)
model_knn

Training K-Nearest Neighbors model...


,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [65]:
print("Training Decision Tree model...")
model_dt.fit(X_train_cv, y_train)
model_dt

Training Decision Tree model...


,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [66]:
# 3. Generate predictions on test set
y_pred_nb = model_nb.predict(X_test_cv)
y_pred_knn = model_knn.predict(X_test_cv)
y_pred_dt = model_dt.predict(X_test_cv)

print("\n=== MODEL TRAINING COMPLETED ===")
print("All three models (Naive Bayes, KNN, Decision Tree) trained successfully!")



=== MODEL TRAINING COMPLETED ===
All three models (Naive Bayes, KNN, Decision Tree) trained successfully!


### =========================================================
### STEP 7: MODEL EVALUATION & COMPARISON
### =========================================================

In [67]:
import pandas as pd
from sklearn.metrics import accuracy_score

In [68]:
# Calculate accuracy for each model
acc_nb = accuracy_score(y_test, y_pred_nb)
acc_knn = accuracy_score(y_test, y_pred_knn)
acc_dt = accuracy_score(y_test, y_pred_dt)

#Print the scores
print("Naive Bayes Accuracy: ", acc_nb)
print("KNN Accuracy: ", acc_knn)
print("Decision Tree Accuracy: ", acc_dt)

Naive Bayes Accuracy:  0.9622494432071269
KNN Accuracy:  0.9037861915367483
Decision Tree Accuracy:  0.9971046770601336


In [69]:
import pandas as pd 
data = { 
    "Model Name": ["Multinomial Naive Bayes", "K-Nearest Neighbors", "Decision Tree"], 
    "Accuracy Score": [acc_nb, acc_knn, acc_dt] 
} 

results_table = pd.DataFrame(data) 
results_table

,Model Name,Accuracy Score
0,Multinomial Naive Bayes,0.962249
1,K-Nearest Neighbors,0.903786
2,Decision Tree,0.997105


In [70]:
sample_check = pd.DataFrame({
    "Actual Label": y_test[:5],
    "Predicted Label": y_pred_nb[:5]
})

sample_check

,Actual Label,Predicted Label
0,1,1
1,1,1
2,0,0
3,1,1
4,1,1


In [71]:
sample_check_dt = pd.DataFrame({
    "Actual Label": y_test[:5],
    "Predicted Label": y_pred_dt[:5]
})

sample_check_dt

,Actual Label,Predicted Label
0,1,1
1,1,1
2,0,0
3,1,1
4,1,1


In [72]:
new_news = ["Breaking: Scientists discover a new planet near Earth!"]
new_news_clean = [clean_text(text) for text in new_news]

new_news_vector = cv.transform(new_news_clean)
prediction = model_dt.predict(new_news_vector)

if prediction[0] == 1: 
    print("Result: True / Real News 📰") 
else: print("Result: Fake News ⚠️")

Result: Fake News ⚠️


In [73]:
new_news = ["FBI Russia probe helped by Australian diplomat tip-off: NYT"]
new_news_clean = [clean_text(text) for text in new_news]

new_news_vector = cv.transform(new_news_clean)
prediction = model_nb.predict(new_news_vector)

if prediction[0] == 1: 
    print("Result: True / Real News 📰") 
else: 
    print("Result: Fake News ⚠️")

Result: True / Real News 📰



### =========================================================
### STEP 8: MODEL PERSISTENCE & GRADIO WEB DEPLOYMENT
### =========================================================

In [74]:
import pickle
import gradio as gr
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
import re

In [75]:
with open("model.pkl", "wb") as model_file:
    pickle.dump(model_nb, model_file)

with open("vectorizer.pkl", "wb") as vec_file:
    pickle.dump(cv, vec_file)

print("Saved 'model.pkl' and 'vectorizer.pkl' successfully!")



Saved 'model.pkl' and 'vectorizer.pkl' successfully!


In [76]:
with open("model.pkl", "rb") as model_file:
    loaded_model = pickle.load(model_file)

with open("vectorizer.pkl", "rb") as vec_file:
    loaded_vectorizer = pickle.load(vec_file)


def predict_fake_news(news_text):
    if not news_text or news_text.strip() == "":
        return "Please enter a valid news headline or article."
    
    # Regex Preprocessing
    cleaned_text = news_text.lower()
    cleaned_text = re.sub(r'[^a-z\s]', '', cleaned_text)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    
    # Vectorization
    text_vector = loaded_vectorizer.transform([cleaned_text])
    
    # Model Prediction
    prediction = loaded_model.predict(text_vector)[0]
    
    # Map numerical prediction (0 -> Fake, 1 -> Real)
    if prediction == 1:
        return "📰 REAL / TRUE NEWS (Authentic Reporting)"
    else:
        return "⚠️ FAKE NEWS (Potentially Misleading / Unverified)"


# --- PART 3: GRADIO WEB INTERFACE ---

# Build Gradio UI
interface = gr.Interface(
    fn=predict_fake_news,
    inputs=gr.Textbox(lines=5, placeholder="Paste a news article body or headline here..."),
    outputs=gr.Textbox(label="Detection Result"),
    title="📰 Fake News Detection System",
    description="An NLP Machine Learning Application powered by CountVectorizer and Naive Bayes Classification.",
    examples=[
        ["WASHINGTON (Reuters) - The U.S. Senate passed the bipartisan infrastructure funding bill today."],
        ["BREAKING: Secret alien spacecraft discovered hidden in the Nevada desert last night!"]
    ]
)


if __name__ == "__main__":
    print("\nLaunching Gradio Web App...")
    interface.launch(share=True)



Launching Gradio Web App...
* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
